#  Create Views and Governance Notes

In this milestone, we create **analyst- or dashboard-friendly views** from the Gold tables.  
We also document **where tables and views live**, and how **Unity Catalog** organizes them for governance.

In [0]:
# Catalog and schema
catalog = "vattenfall_dev"
gold_schema = "gold"

# Gold tables
gold_dashboard = f"{catalog}.{gold_schema}.gold_regional_operations_dashboard"
gold_market = f"{catalog}.{gold_schema}.gold_daily_market_summary"
gold_weather = f"{catalog}.{gold_schema}.gold_weather_impact_summary"

# Views we want to create
vw_dashboard = f"{catalog}.{gold_schema}.vw_regional_operations_dashboard"
vw_market_weather = f"{catalog}.{gold_schema}.vw_market_weather_overview"

## Step 1: Create a dashboard view

This view presents **regional operational metrics** with key KPIs:

- `incident_count`
- `elevated_incident_count`
- `weather_risk_signal`
- `avg_market_price`

In [0]:
%sql
-- Create or replace view for regional operations dashboard
CREATE OR REPLACE VIEW vattenfall_dev.gold.vw_regional_operations_dashboard AS
SELECT
    region,
    report_day,
    incident_count,
    elevated_incident_count,
    weather_risk_signal,
    avg_market_price
FROM vattenfall_dev.gold.gold_regional_operations_dashboard;

## Step 2: Create a market + weather overview view

This view **joins market summary and weather summary** by `region` and `report_day` to help analysts explore correlations:

- average market price
- maximum market price
- total market volume
- high-price count
- avg temperature
- wind speed
- precipitation
- weather risk label

In [0]:
%sql
CREATE OR REPLACE VIEW vattenfall_dev.gold.vw_market_weather_overview AS
SELECT
    m.region,
    m.report_day,
    m.avg_market_price,
    m.max_market_price,
    m.total_market_volume,
    m.high_price_count,
    w.avg_temperature_c,
    w.avg_wind_speed_kmh,
    w.total_precipitation_mm,
    w.weather_risk_label
FROM vattenfall_dev.gold.gold_daily_market_summary m
LEFT JOIN vattenfall_dev.gold.gold_weather_impact_summary w
  ON m.region = w.region
 AND m.report_day = w.report_day;
 

## Step 3: Governance Notes

- **Tables**: All Gold tables live in `vattenfall_dev.gold`.  
- **Views**: Analyst-facing views live in the same schema for easy access.  
- **Unity Catalog**: Organizes tables and views by catalog + schema. Enables:  
  - clear ownership
  - easy discovery
  - row/column-level permissions if needed  

In a full production environment, you would grant **SELECT access** to analysts, **WRITE/UPDATE** to ETL pipelines, and potentially restrict sensitive columns using **column masking** or **row access policies**.

In [0]:
%sql
SELECT * FROM vattenfall_dev.gold.vw_regional_operations_dashboard LIMIT 5;


In [0]:
%sql
SELECT * FROM vattenfall_dev.gold.vw_market_weather_overview LIMIT 5;